# 🧹 Clase 5 — Datos reales = datos sucios
**Maestría en Fintech — Programación para el Análisis de Datos**

**Pregunta que responde:** *Mi dataset tiene nulos, duplicados y valores raros. ¿Qué hago antes de sacar conclusiones?*

> 💡 Un análisis es tan bueno como la calidad de sus datos. Y el analista es responsable de garantizarla.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('✅ Listo')

In [ ]:
# Los datos se descargan solos desde el repo del curso.
# No hace falta subir ningún archivo a Colab.
DATOS = 'https://raw.githubusercontent.com/camilojaure/itba-pad/main/datasets/'

# Cargamos el dataset 'sucio'
df = pd.read_csv(DATOS + 'transacciones_sucias.csv')
print(f'Dataset cargado: {len(df):,} filas, {df.shape[1]} columnas')
df.head(10)


## 1. Diagnóstico inicial — El checklist del analista

Antes de limpiar nada, necesitás entender qué tan sucio está el dataset.

In [ ]:
# ── PASO 1: Forma general
print(f'Dimensiones: {df.shape}')
print(f'Tipos de datos:')
print(df.dtypes)
print(f'\nMemoria: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')

In [ ]:
# ── PASO 2: Valores nulos
nulos = df.isnull().sum()
nulos_pct = (df.isnull().mean() * 100).round(2)
resumen_nulos = pd.DataFrame({'cantidad': nulos, 'porcentaje': nulos_pct})
resumen_nulos = resumen_nulos[resumen_nulos['cantidad'] > 0].sort_values('porcentaje', ascending=False)
print('Columnas con valores nulos:')
resumen_nulos

In [ ]:
# Heatmap de nulos — visualización muy útil
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Mapa de valores nulos (amarillo = nulo)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── PASO 3: Duplicados
duplicados = df.duplicated().sum()
print(f'Filas duplicadas: {duplicados:,} ({duplicados/len(df)*100:.2f}%)')
df[df.duplicated()].head(5)

In [ ]:
# ── PASO 4: Valores únicos en columnas categóricas
# Acá aparecen las inconsistencias de texto
for col in ['categoria', 'tipo']:
    print(f'\nValores únicos en "{col}":')
    print(df[col].value_counts().to_string())

In [ ]:
# ── PASO 5: Outliers en columnas numéricas
print('Estadísticas de monto_ars:')
df['monto_ars'].describe().round(2)

In [ ]:
# Boxplot para ver outliers visualmente
plt.figure(figsize=(10, 4))
sns.boxplot(x=df['monto_ars'].dropna(), color='salmon')
plt.title('Distribución de montos — ¿Ves los outliers?', fontsize=13)
plt.xlabel('Monto ARS')
plt.tight_layout()
plt.show()

## 2. Limpieza — Paso a paso

### 2.1 Eliminar duplicados

In [ ]:
df_limpio = df.copy()

antes = len(df_limpio)
df_limpio = df_limpio.drop_duplicates()
despues = len(df_limpio)
print(f'Duplicados eliminados: {antes - despues:,}')
print(f'Filas restantes: {len(df_limpio):,}')

### 2.2 Tratar valores nulos

> La decisión de qué hacer con un nulo NO es técnica — es de negocio. Un nulo en 'monto' no es lo mismo que un nulo en 'categoría'.

In [ ]:
# Nulos en 'fecha' → eliminar (no podemos imputar una fecha de transacción)
df_limpio = df_limpio.dropna(subset=['fecha'])
print(f'Filas después de eliminar nulos en fecha: {len(df_limpio):,}')

# Nulos en 'monto_ars' → eliminar (no tiene sentido una transacción sin monto)
df_limpio = df_limpio.dropna(subset=['monto_ars'])
print(f'Filas después de eliminar nulos en monto: {len(df_limpio):,}')

# Nulos en 'categoria' → imputar con 'Sin categoría'
df_limpio['categoria'] = df_limpio['categoria'].fillna('Sin categoría')
print(f'Nulos en categoría: {df_limpio["categoria"].isnull().sum()}')

### 2.3 Corregir inconsistencias en texto

In [ ]:
# Normalizar categoria y tipo a lowercase con strip
df_limpio['categoria'] = df_limpio['categoria'].str.strip().str.title()
df_limpio['tipo'] = df_limpio['tipo'].str.strip().str.lower()

# Reemplazar variantes de 'crédito'
df_limpio['tipo'] = df_limpio['tipo'].replace({'credito': 'crédito', 'credit': 'crédito'})

print('Valores únicos en tipo:', df_limpio['tipo'].unique())
print('Cantidad de categorías:', df_limpio['categoria'].nunique())

### 2.4 Tratar outliers

> Un outlier no siempre es un error — puede ser una transacción legítimamente grande. El criterio es de negocio.

In [ ]:
# Identificar outliers con el método IQR
Q1 = df_limpio['monto_ars'].quantile(0.25)
Q3 = df_limpio['monto_ars'].quantile(0.75)
IQR = Q3 - Q1
limite_inf = Q1 - 3 * IQR
limite_sup = Q3 + 3 * IQR

outliers = df_limpio[(df_limpio['monto_ars'] < limite_inf) | (df_limpio['monto_ars'] > limite_sup)]
print(f'Outliers detectados: {len(outliers):,}')
print(f'Límite inferior: ${limite_inf:,.0f}')
print(f'Límite superior: ${limite_sup:,.0f}')
outliers[['transaccion_id','monto_ars','categoria']].head(10)

In [ ]:
# Eliminamos transacciones con monto negativo o cero (errores claros)
df_limpio = df_limpio[df_limpio['monto_ars'] > 0]

# Winsorizing: capamos el percentil 99 en lugar de eliminar
p99 = df_limpio['monto_ars'].quantile(0.99)
df_limpio['monto_ars'] = df_limpio['monto_ars'].clip(upper=p99)

print(f'Dataset limpio final: {len(df_limpio):,} filas')
print(f'Monto máximo después del cap: ${df_limpio["monto_ars"].max():,.0f}')

## 3. Comparación antes vs después

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Antes
axes[0].hist(df['monto_ars'].dropna().clip(-1000, 500000), bins=50, color='salmon', edgecolor='white')
axes[0].set_title(f'ANTES — {len(df):,} filas', fontsize=13)
axes[0].set_xlabel('Monto ARS')

# Después
axes[1].hist(df_limpio['monto_ars'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_title(f'DESPUÉS — {len(df_limpio):,} filas', fontsize=13)
axes[1].set_xlabel('Monto ARS')

plt.suptitle('Distribución de montos: antes y después de la limpieza', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'\nResumen de la limpieza:')
print(f'  Filas originales: {len(df):,}')
print(f'  Filas después de limpiar: {len(df_limpio):,}')
print(f'  Filas removidas: {len(df)-len(df_limpio):,} ({(len(df)-len(df_limpio))/len(df)*100:.1f}%)')

In [ ]:
# Guardamos el dataset limpio
df_limpio.to_csv('transacciones_limpias.csv', index=False)
print('✅ Dataset limpio guardado como transacciones_limpias.csv')

---
## 🧑‍💻 Práctica — Tu turno

In [ ]:
# EJERCICIO 1
# Calculá el % de nulos de CADA columna del dataset original.
# ¿Cuál columna tiene más nulos?
# Tu código acá:


In [ ]:
# EJERCICIO 2
# ¿Cuántos duplicados exactos hay? Mostralos.
# Tu código acá:


In [ ]:
# EJERCICIO 3
# ¿Hay transacciones con monto = 0? ¿Y con monto negativo? ¿Cuántas?
# Tu código acá:


In [ ]:
# EJERCICIO 4
# Normalizá la columna 'categoria' del dataset original para que todas las variantes
# de la misma categoría queden escritas igual (ej: 'SUPERMERCADO' → 'Supermercado')
# Tu código acá:


In [ ]:
# EJERCICIO 5
# Imputá los valores nulos de 'monto_ars' con la mediana del monto de su categoría.
# Tip: groupby('categoria')['monto_ars'].transform('median')
# Tu código acá:


In [ ]:
# EJERCICIO 6
# Calculá el monto en el percentil 95 y 99. ¿Cuántas transacciones están por encima del p99?
# Tu código acá:


In [ ]:
# EJERCICIO 7
# Usando el dataset limpio, graficá la distribución de montos por categoría
# usando un boxplot (sns.boxplot).
# Tu código acá:


In [ ]:
# EJERCICIO 8
# Validá el dataset limpio: mostrá un resumen de cuántos nulos quedan, 
# cuántos duplicados, y los valores únicos de 'tipo' y 'categoria'.
# Tu código acá:


In [ ]:
# EJERCICIO 9
# ¿Cambia el monto promedio por categoría entre el dataset sucio y el limpio?
# Mostrá la comparación en una tabla.
# Tu código acá:


In [ ]:
# EJERCICIO 10 — Desafío
# Escribí una función llamada 'reporte_calidad(df)' que reciba un DataFrame
# y devuelva un resumen de: nulos por columna, duplicados totales, y outliers en columnas numéricas.
# Es tu herramienta reutilizable de data quality.
# Tu código acá:

def reporte_calidad(df):
    # Tu código acá
    pass

# Probala:
# reporte_calidad(df)
